# LeetCode 376 — Wiggle Subsequence bằng Python

Link đề: https://leetcode.com/problems/wiggle-subsequence/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def wiggleMaxLength(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề được viết lại bằng tiếng Việt theo đề gốc trên LeetCode; mọi đáp án đều đã được kiểm chứng bằng code, kể cả bằng vét cạn mọi dãy con.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **dãy lắc (wiggle)** là dãy mà hiệu giữa các phần tử liên tiếp **đổi dấu liên tục** (dương, âm, dương, ... hoặc âm, dương, âm, ...); cần tìm **dãy con** (không nhất thiết liên tiếp) lắc **dài nhất**.
2. Nắm các công cụ Python: **dãy con** khác **dãy con liên tiếp (subarray)**, tính hiệu bằng `zip(nums, nums[1:])`, hàm dấu `(a > b) - (a < b)`, vét cạn bằng **bitmask**, hai biến trạng thái `up`/`down`.
3. Đi từ **vét cạn mọi dãy con `O(2^n · n)`** → **DP `O(n²)` với hai mảng `up[i]`, `down[i]`** → **DP rút gọn `O(n)` chỉ cần hai biến** → **tham lam: đếm số lần đổi chiều (đỉnh/đáy)**.
4. Hiểu mấu chốt: chỉ cần giữ **cực trị cục bộ** (đỉnh và đáy) của dãy; mọi đoạn tăng/giảm liên tục đều "nén" lại chỉ còn đầu mút.
5. Tránh lỗi kinh điển: nhầm dãy con với dãy con liên tiếp, đếm hiệu bằng `0` là đổi chiều, cập nhật `prev_diff` lúc hiệu `0`, quên trường hợp `n = 1` hoặc toàn số bằng nhau.

## 2. Đề bài & ví dụ

Một **dãy lắc** là dãy số mà các hiệu giữa các số liên tiếp **luân phiên** dương và âm một cách **chặt** (không có hiệu bằng `0`). Hiệu đầu tiên có thể dương hoặc âm. Dãy một phần tử, và dãy hai phần tử **khác nhau**, được coi là dãy lắc.

- `[1, 7, 4, 9, 2, 5]` là dãy lắc: hiệu `(6, -3, 5, -7, 3)` đổi dấu liên tục.
- `[1, 4, 7, 2, 5]` **không**: hai hiệu đầu cùng dương.
- `[1, 7, 4, 5, 5]` **không**: hiệu cuối bằng `0`.

**Dãy con** nhận được bằng cách **xóa** một số phần tử (có thể không xóa) và giữ nguyên thứ tự còn lại.

Cho mảng `nums`, trả về **độ dài** dãy con lắc **dài nhất**.

**Ví dụ 1:** `nums = [1,7,4,9,2,5]` → `6` (cả dãy).

**Ví dụ 2:** `nums = [1,17,5,10,13,15,10,5,16,8]` → `7` (ví dụ `[1,17,10,13,10,16,8]` với hiệu `(16,-7,3,-3,6,-8)`).

**Ví dụ 3:** `nums = [1,2,3,4,5,6,7,8,9]` → `2`.

**Ràng buộc:** `1 <= nums.length <= 1000`, `0 <= nums[i] <= 1000`.

**Follow up:** giải trong `O(n)` thời gian?

In [1]:
import itertools
import random
import time
from typing import List

examples = [
    ([1, 7, 4, 9, 2, 5], 6),
    ([1, 17, 5, 10, 13, 15, 10, 5, 16, 8], 7),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9], 2),
]
for nums, expected in examples:
    print(nums, "->", expected)

[1, 7, 4, 9, 2, 5] -> 6
[1, 17, 5, 10, 13, 15, 10, 5, 16, 8] -> 7
[1, 2, 3, 4, 5, 6, 7, 8, 9] -> 2


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) khác dãy con liên tiếp (subarray)

- **Subarray:** các phần tử **liền nhau**, như `nums[2:5]`.
- **Subsequence:** chọn phần tử **tùy ý theo thứ tự**, có thể bỏ cách quãng.

Với `nums = [10, 20, 30, 40]`, `[10, 30, 40]` là dãy con (bỏ `20`) nhưng **không** là subarray.

In [2]:
nums = [10, 20, 30, 40]
print("subarray :", nums[1:4])                       # [20, 30, 40]
chon = [0, 2, 3]                                      # chọn các chỉ số 0, 2, 3
print("subsequence:", [nums[i] for i in chon])        # [10, 30, 40]

subarray : [20, 30, 40]
subsequence: [10, 30, 40]


### 3.2. Hiệu liên tiếp bằng `zip(nums, nums[1:])`

`nums[1:]` là mảng bỏ phần tử đầu. `zip(a, b)` ghép từng cặp cùng vị trí và dừng khi một bên hết. Vậy `zip(nums, nums[1:])` cho ra các cặp **liền kề** `(nums[0], nums[1])`, `(nums[1], nums[2])`, ...

In [3]:
nums = [1, 7, 4, 9, 2, 5]
print(list(zip(nums, nums[1:])))                      # [(1, 7), (7, 4), (4, 9), (9, 2), (2, 5)]
diffs = [b - a for a, b in zip(nums, nums[1:])]
print(diffs)                                          # [6, -3, 5, -7, 3]

[(1, 7), (7, 4), (4, 9), (9, 2), (2, 5)]
[6, -3, 5, -7, 3]


### 3.3. Dấu của một số và kiểm tra "luân phiên"

Ta chỉ quan tâm dấu của hiệu: `+`, `-` hay `0`. Mẹo viết hàm dấu ngắn gọn: `(x > 0) - (x < 0)` (trong Python `True` là `1`, `False` là `0`) cho ra `1`, `-1` hoặc `0`.

Dãy hiệu là **lắc** khi: không có `0`, và **hai hiệu liền nhau có dấu ngược nhau**, tức tích `d[i] * d[i+1] < 0`.

In [4]:
def sign(x):
    return (x > 0) - (x < 0)

print([sign(x) for x in (6, -3, 0, 12, -9)])          # [1, -1, 0, 1, -1]

def is_wiggle(seq):
    """Kiểm tra dãy lắc theo định nghĩa đề (độ dài 1, hoặc 2 phần tử khác nhau, đều lắc)."""
    d = [b - a for a, b in zip(seq, seq[1:])]
    if any(x == 0 for x in d):
        return False
    return all(d[i] * d[i + 1] < 0 for i in range(len(d) - 1))

print(is_wiggle([1, 7, 4, 9, 2, 5]), is_wiggle([1, 4, 7, 2, 5]), is_wiggle([1, 7, 4, 5, 5]),
      is_wiggle([5]), is_wiggle([3, 3]), is_wiggle([3, 4]))

[1, -1, 0, 1, -1]
True False False True False True


### 3.4. Vét cạn dãy con bằng bitmask

Mảng `n` phần tử có `2^n` dãy con. Dùng số nguyên `mask` từ `0` đến `2^n - 1`: bit `i` bằng `1` nghĩa là giữ `nums[i]`. `mask >> i & 1` lấy bit `i`.

In [5]:
nums = [3, 1, 2]
for mask in range(1, 1 << len(nums)):                  # bỏ mask = 0 (dãy rỗng)
    sub = [nums[i] for i in range(len(nums)) if mask >> i & 1]
    print(f"{mask:03b} -> {sub}")

001 -> [3]
010 -> [1]
011 -> [3, 1]
100 -> [2]
101 -> [3, 2]
110 -> [1, 2]
111 -> [3, 1, 2]


### 3.5. Hai biến trạng thái (máy trạng thái nhỏ)

Khi duyệt mảng, ta có thể chỉ nhớ **hai con số**: `up` và `down`, thay cho cả mảng. Đây là kỹ thuật **rút gọn DP** quen thuộc: nếu `dp[i]` chỉ phụ thuộc `dp[i-1]` thì chỉ cần giữ giá trị vòng trước. Ở đây ta sẽ định nghĩa `up`/`down` rõ ràng ở mục 7.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hình dung dãy số là một đường **gấp khúc** lên xuống. Dãy lắc dài nhất = số **đỉnh + đáy** (các điểm đổi chiều) cộng thêm hai đầu.

Lấy `nums = [1, 17, 5, 10, 13, 15, 10, 5, 16, 8]`. Các hiệu: `16, -12, 5, 3, 2, -5, -5, 11, -8`.

- Nhóm các hiệu **cùng dấu liên tiếp** thành một "đoạn": `[16]` (lên), `[-12]` (xuống), `[5,3,2]` (lên), `[-5,-5]` (xuống), `[11]` (lên), `[-8]` (xuống). Có **6 đoạn** → có **5 lần đổi chiều**.
- Chỉ giữ hai đầu của mỗi đoạn: các số `1, 17, 5, 15, 5, 16, 8` → **7** phần tử. Đúng `7`!

Quy tắc trực giác: **trong một đoạn tăng liên tục, ta chỉ cần giữ điểm cuối** (điểm cao nhất của đoạn). Giữ điểm giữa không giúp thêm phần tử nào mà còn có thể làm hỏng sự luân phiên. Vậy:

**Đáp án = 1 + (số đoạn cùng chiều liên tiếp)** (bỏ qua các hiệu bằng `0`), tức = 1 + số lần đổi dấu + 1 (nếu có ít nhất một hiệu khác 0).

In [6]:
nums = [1, 17, 5, 10, 13, 15, 10, 5, 16, 8]
diffs = [b - a for a, b in zip(nums, nums[1:])]
print("hiệu :", diffs)
dau = [sign(d) for d in diffs]
print("dấu  :", dau)
doan = 1 + sum(1 for a, b in zip(dau, dau[1:]) if a != b)           # số đoạn cùng dấu liên tiếp
print("số đoạn:", doan, "-> độ dài dãy lắc dài nhất =", doan + 1)     # 7

hiệu : [16, -12, 5, 3, 2, -5, -5, 11, -8]
dấu  : [1, -1, 1, 1, 1, -1, -1, 1, -1]
số đoạn: 6 -> độ dài dãy lắc dài nhất = 7


## 5. Cách 1 — Brute Force / baseline: duyệt mọi dãy con

**Ý tưởng:** thử **mọi** dãy con khác rỗng, kiểm tra từng dãy có lắc không bằng `is_wiggle`, lấy độ dài lớn nhất. Dùng bitmask (mục 3.4).

Chạy ví dụ nhỏ (ví dụ 3 có 9 phần tử → `2^9 = 512` dãy con, nhanh).

In [7]:
def wiggle_brute(nums):
    n = len(nums)
    best = 0
    for mask in range(1, 1 << n):
        sub = [nums[i] for i in range(n) if mask >> i & 1]
        if len(sub) > best and is_wiggle(sub):
            best = len(sub)
    return best

for nums, expected in examples:
    got = wiggle_brute(nums)
    print(nums, got, expected)
    assert got == expected

[1, 7, 4, 9, 2, 5] 6 6
[1, 17, 5, 10, 13, 15, 10, 5, 16, 8] 7 7
[1, 2, 3, 4, 5, 6, 7, 8, 9] 2 2


**Độ phức tạp:** có `2^n` dãy con, mỗi dãy kiểm tra `O(n)` → `O(2^n · n)` thời gian, `O(n)` bộ nhớ. Với `n = 1000` là **không thể**.

**Nút thắt:** ta kiểm tra lại từng dãy từ đầu, bỏ qua cấu trúc: "dãy lắc dài nhất kết thúc ở một vị trí" có thể tái sử dụng khi thêm phần tử mới.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Cấu trúc con:** một dãy lắc kết thúc tại `nums[i]` thì phần còn lại (bỏ `nums[i]`) cũng là dãy lắc kết thúc tại một `nums[j]` nào đó (`j < i`) với hiệu **cuối ngược chiều**. Vậy chỉ cần biết: *dãy lắc dài nhất kết thúc tại `j` với hiệu cuối **dương** (`up`) hoặc **âm** (`down`)*.
2. **Hai trạng thái:** `up[i]` = dài nhất kết thúc tại `i` mà bước cuối **đi lên**; `down[i]` = bước cuối **đi xuống**. Mở rộng:
   - nếu `nums[i] > nums[j]` thì bước `j → i` đi lên, ta nối vào dãy kết thúc bằng bước **xuống**: `up[i] = max(up[i], down[j] + 1)`.
   - nếu `nums[i] < nums[j]` thì `down[i] = max(down[i], up[j] + 1)`.
3. **Tối giản:** vì có thể "đổi cực trị" (thay phần tử cuối bằng phần tử tiếp theo cùng chiều không làm xấu đi), chỉ cần `up`/`down` **tại vị trí `i - 1`** → `O(n)`.
4. **Tham lam:** tương đương đếm số lần đổi chiều, như mục 4.

## 7. Các cách tiếp theo

### Cách 2 — DP `O(n²)` với hai mảng `up`, `down`

**Giải thích từng đoạn:**

- `up = [1] * n`, `down = [1] * n`: mỗi phần tử đứng một mình là dãy lắc độ dài `1`.
- Với mỗi `i`, duyệt mọi `j < i`: nếu `nums[i] > nums[j]` → `up[i] = max(up[i], down[j] + 1)`; nếu `nums[i] < nums[j]` → `down[i] = max(down[i], up[j] + 1)`; bằng nhau thì bỏ qua.
- Đáp án = `max` của tất cả `up` và `down`.

In [8]:
def wiggle_dp_n2(nums):
    n = len(nums)
    up = [1] * n
    down = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[i] > nums[j]:
                up[i] = max(up[i], down[j] + 1)
            elif nums[i] < nums[j]:
                down[i] = max(down[i], up[j] + 1)
    return max(max(up), max(down))

for nums, expected in examples:
    got = wiggle_dp_n2(nums)
    print(nums, got, expected)
    assert got == expected

[1, 7, 4, 9, 2, 5] 6 6
[1, 17, 5, 10, 13, 15, 10, 5, 16, 8] 7 7
[1, 2, 3, 4, 5, 6, 7, 8, 9] 2 2


In [9]:
class SolutionN2:
    def wiggleMaxLength(self, nums: List[int]) -> int:
        n = len(nums)
        up = [1] * n
        down = [1] * n
        for i in range(n):
            for j in range(i):
                if nums[i] > nums[j]:
                    up[i] = max(up[i], down[j] + 1)
                elif nums[i] < nums[j]:
                    down[i] = max(down[i], up[j] + 1)
        return max(max(up), max(down))

print(SolutionN2().wiggleMaxLength([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]))

7


**Độ phức tạp:** `O(n²)` thời gian (hai vòng lặp), `O(n)` bộ nhớ. Với `n = 1000` là khoảng `500.000` phép — nộp được, nhưng follow up muốn `O(n)`.

### Cách 3 — DP rút gọn `O(n)` với hai biến `up`, `down`

**Ý tưởng:** `up` = độ dài dãy lắc dài nhất **đã duyệt xong** mà bước cuối đi **lên**; `down` tương tự cho bước **xuống**. Khi gặp `nums[i]` so với `nums[i-1]`:

- `nums[i] > nums[i-1]` (đi lên): bước lên mới nối vào dãy kết thúc bằng bước xuống → `up = down + 1`. (`down` không đổi.)
- `nums[i] < nums[i-1]` (đi xuống): `down = up + 1`.
- Bằng nhau: không đổi gì.

Vì sao chỉ cần so với `nums[i-1]` mà không so với mọi `j < i`? Vì nếu dãy con tối ưu kết thúc tại `j < i-1` thì ta luôn có thể "kéo dài" chọn phần tử cực trị nằm giữa, và giá trị `up`/`down` tại `i-1` đã **bao gồm** lựa chọn đó (xem mục 10).

In [10]:
def wiggle_updown(nums):
    up = down = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            up = down + 1
        elif nums[i] < nums[i - 1]:
            down = up + 1
    return max(up, down)

for nums, expected in examples:
    got = wiggle_updown(nums)
    print(nums, got, expected)
    assert got == expected

[1, 7, 4, 9, 2, 5] 6 6
[1, 17, 5, 10, 13, 15, 10, 5, 16, 8] 7 7
[1, 2, 3, 4, 5, 6, 7, 8, 9] 2 2


In [11]:
class SolutionUpDown:
    def wiggleMaxLength(self, nums: List[int]) -> int:
        up = down = 1
        for i in range(1, len(nums)):
            if nums[i] > nums[i - 1]:
                up = down + 1
            elif nums[i] < nums[i - 1]:
                down = up + 1
        return max(up, down)

print(SolutionUpDown().wiggleMaxLength([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]))

7


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ. Đây là lời giải thỏa follow up.

### Cách 4 — Tham lam: đếm số lần đổi chiều (đỉnh/đáy)

**Ý tưởng (mục 4):** giữ lại **hiệu khác 0 gần nhất** `prev_diff`. Mỗi khi gặp hiệu `diff` mới mà **ngược dấu** với `prev_diff` (hoặc là hiệu khác 0 đầu tiên) thì ta có thêm một phần tử vào dãy. Quan trọng: **chỉ cập nhật `prev_diff` khi có đổi chiều** (xem lỗi 3 ở mục 13).

**Giải thích từng đoạn:**

- `if len(nums) < 2: return len(nums)`.
- `ans = 1`, `prev = 0` (chưa có hiệu nào).
- Với mỗi `i`: `diff = nums[i] - nums[i-1]`. Nếu `(diff > 0 and prev <= 0) or (diff < 0 and prev >= 0)`: `ans += 1` và `prev = diff`.
- Điều kiện `prev <= 0` với `diff > 0`: bao cả `prev < 0` (đổi chiều) và `prev == 0` (hiệu khác 0 đầu tiên).

In [12]:
def wiggle_greedy(nums):
    if len(nums) < 2:
        return len(nums)
    ans = 1
    prev = 0
    for i in range(1, len(nums)):
        diff = nums[i] - nums[i - 1]
        if (diff > 0 and prev <= 0) or (diff < 0 and prev >= 0):
            ans += 1
            prev = diff
    return ans

for nums, expected in examples:
    got = wiggle_greedy(nums)
    print(nums, got, expected)
    assert got == expected

[1, 7, 4, 9, 2, 5] 6 6
[1, 17, 5, 10, 13, 15, 10, 5, 16, 8] 7 7
[1, 2, 3, 4, 5, 6, 7, 8, 9] 2 2


In [13]:
class SolutionGreedy:
    def wiggleMaxLength(self, nums: List[int]) -> int:
        if len(nums) < 2:
            return len(nums)
        ans = 1
        prev = 0
        for i in range(1, len(nums)):
            diff = nums[i] - nums[i - 1]
            if (diff > 0 and prev <= 0) or (diff < 0 and prev >= 0):
                ans += 1
                prev = diff
        return ans

print(SolutionGreedy().wiggleMaxLength([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]))

7


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

Lưu ý: Cách 3 và Cách 4 thực chất là **hai góc nhìn** của cùng một sự thật (đếm số lần đổi chiều), nhưng cách làm khác nhau: Cách 3 đi từ định nghĩa DP, Cách 4 đi từ hình học "đỉnh/đáy".

## 8. TRACE từng bước

Ta trace **Cách 3 (up/down)** với `nums = [1, 17, 5, 10, 13, 15, 10, 5, 16, 8]`.

| i | nums[i-1] → nums[i] | hướng | up | down |
|---|---|---|---|---|
| 0 | khởi tạo | | 1 | 1 |
| 1 | 1 → 17 | lên | `down+1 = 2` | 1 |
| 2 | 17 → 5 | xuống | 2 | `up+1 = 3` |
| 3 | 5 → 10 | lên | `down+1 = 4` | 3 |
| 4 | 10 → 13 | lên | `down+1 = 4` | 3 |
| 5 | 13 → 15 | lên | `down+1 = 4` | 3 |
| 6 | 15 → 10 | xuống | 4 | `up+1 = 5` |
| 7 | 10 → 5 | xuống | 4 | `up+1 = 5` |
| 8 | 5 → 16 | lên | `down+1 = 6` | 5 |
| 9 | 16 → 8 | xuống | 6 | `up+1 = 7` |

Đáp án `max(6, 7) = 7`. Chú ý: nhiều bước lên liên tiếp (`i = 3, 4, 5`) **không làm `up` tăng thêm**.

In [14]:
def trace_updown(nums):
    up = down = 1
    print(f"{'i':>2} {'prev→cur':>10} {'hướng':>6} {'up':>3} {'down':>5}")
    print(f"{0:>2} {'khởi tạo':>10} {'':>6} {up:>3} {down:>5}")
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if b > a:
            up = down + 1
            h = "lên"
        elif b < a:
            down = up + 1
            h = "xuống"
        else:
            h = "bằng"
        print(f"{i:>2} {f'{a}→{b}':>10} {h:>6} {up:>3} {down:>5}")
    print("Đáp án =", max(up, down))
    return max(up, down)

assert trace_updown([1, 17, 5, 10, 13, 15, 10, 5, 16, 8]) == 7

 i   prev→cur  hướng  up  down
 0   khởi tạo          1     1
 1       1→17    lên   2     1
 2       17→5  xuống   2     3
 3       5→10    lên   4     3
 4      10→13    lên   4     3
 5      13→15    lên   4     3
 6      15→10  xuống   4     5
 7       10→5  xuống   4     5
 8       5→16    lên   6     5
 9       16→8  xuống   6     7
Đáp án = 7


In [15]:
# Trace thêm Cách 4 (tham lam) để thấy "prev" chỉ đổi khi đổi chiều, kể cả với số bằng nhau
def trace_greedy(nums):
    ans, prev = 1, 0
    print(f"{'i':>2} {'diff':>5} {'prev':>5} {'đổi chiều?':>11} {'ans':>4}")
    for i in range(1, len(nums)):
        diff = nums[i] - nums[i - 1]
        doi = (diff > 0 and prev <= 0) or (diff < 0 and prev >= 0)
        if doi:
            ans += 1
            prev = diff
        print(f"{i:>2} {diff:>5} {prev:>5} {str(doi):>11} {ans:>4}")
    return ans

assert trace_greedy([1, 2, 2, 2, 3, 1, 1, 4]) == 4

 i  diff  prev  đổi chiều?  ans
 1     1     1        True    2
 2     0     1       False    2
 3     0     1       False    2
 4     1     1       False    2
 5    -2    -2        True    3
 6     0    -2       False    3
 7     3     3        True    4


Với `[1, 2, 2, 2, 3, 1, 1, 4]`: các hiệu `1, 0, 0, 1, -2, 0, 3`. Hiệu `0` bị bỏ qua hoàn toàn; hiệu `1` (lần 2) cùng dấu với `prev = 1` nên không đếm. Dãy lắc dài nhất: `1, 3, 1, 4` (độ dài `4`).

## 9. Code nộp LeetCode

Ta nộp Cách 3 (up/down): ngắn, `O(n)`, `O(1)`, dễ chứng minh.

In [16]:
class Solution:
    def wiggleMaxLength(self, nums: List[int]) -> int:
        # up: dài nhất đã thấy mà bước cuối đi lên; down: bước cuối đi xuống
        up = down = 1
        for i in range(1, len(nums)):
            if nums[i] > nums[i - 1]:
                up = down + 1
            elif nums[i] < nums[i - 1]:
                down = up + 1
        return max(up, down)

sol = Solution()
for nums, expected in examples:
    got = sol.wiggleMaxLength(nums)
    print(nums, got, expected, "OK" if got == expected else "SAI")
    assert got == expected

[1, 7, 4, 9, 2, 5] 6 6 OK
[1, 17, 5, 10, 13, 15, 10, 5, 16, 8] 7 7 OK
[1, 2, 3, 4, 5, 6, 7, 8, 9] 2 2 OK


## 10. Vì sao thuật toán đúng

**Định nghĩa.** Sau khi xử lý xong `nums[0..i]`, `up_i` (`down_i`) = độ dài lớn nhất của dãy con lắc của `nums[0..i]` mà bước cuối đi lên (xuống); nếu dãy chỉ có một phần tử, tính vào cả hai.

**Điều cần chứng minh:** công thức cập nhật chỉ dựa trên `nums[i-1]`.

1. Giả sử `nums[i] > nums[i-1]`. Ta cần `up_i = down_{i-1} + 1`.
   - (`≤`): `up_i` là độ dài một dãy lắc có bước cuối lên, kết thúc ở `nums[i]` (hoặc sớm hơn nếu không dùng `nums[i]`). Bỏ phần tử cuối đi, ta được dãy lắc có bước cuối xuống trên `nums[0..i-1]`, nên `up_i ≤ down_{i-1} + 1`. (Nếu không dùng `nums[i]` thì `up_i = up_{i-1} ≤ down_{i-1} + 1` theo giả thiết quy nạp.)
   - (`≥`): lấy dãy lắc dài `down_{i-1}` trên `nums[0..i-1]` có bước cuối xuống, phần tử cuối là `x = nums[m]` (`m ≤ i-1`). Nếu `x < nums[i]` thì nối `nums[i]` ngay, được bước lên. Nếu không (`x ≥ nums[i] > nums[i-1]`) thì `nums[i-1]` nằm **sau** `x` và nhỏ hơn `x`, mà `x` là điểm kết thúc của bước xuống, nên thay `x` bằng `nums[i-1]` (nhỏ hơn hoặc bằng, vẫn đứng sau cùng vị trí hợp lệ) cho ra dãy cũng có bước cuối xuống và có phần tử cuối nhỏ hơn `nums[i]`; sau đó nối `nums[i]`. (Trường hợp `m = i-1` đã xử lý ở nhánh đầu.) Vậy `up_i ≥ down_{i-1} + 1`.
2. Trường hợp `nums[i] < nums[i-1]` đối xứng. Trường hợp bằng nhau: `nums[i]` giống `nums[i-1]`, thêm vào không tạo được dãy dài hơn nên giữ nguyên.

**Kiểm chứng bằng code** (mục 12): so với vét cạn bitmask trên hàng nghìn input ngẫu nhiên nhỏ, và giữa các cách trên input lớn.

**Trực giác hình học:** giữ hai đầu mỗi "đoạn đơn điệu" tạo dãy lắc; không thể có dãy lắc dài hơn số đoạn + 1 vì mỗi phần tử mới phải đổi chiều so với bước trước và mỗi đoạn đơn điệu chứa nhiều nhất một bước thực.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn bitmask | `O(2^n · n)` | `O(n)` |
| 2. DP `up[]`, `down[]` | `O(n²)` | `O(n)` |
| 3. DP hai biến | `O(n)` | `O(1)` |
| 4. Tham lam đếm đổi chiều | `O(n)` | `O(1)` |

Với `n = 1000`: Cách 3/4 chỉ cỡ `1000` bước; Cách 2 cỡ `500.000` bước.

In [17]:
random.seed(376)
big = [random.randint(0, 1000) for _ in range(1000)]
for ten, fn in [("DP n^2", SolutionN2().wiggleMaxLength), ("up/down", SolutionUpDown().wiggleMaxLength), ("greedy", SolutionGreedy().wiggleMaxLength)]:
    t = time.perf_counter()
    r = fn(big)
    print(f"{ten:<8}: kết quả {r}, {1000 * (time.perf_counter() - t):.3f} ms")

DP n^2  : kết quả 664, 40.550 ms
up/down : kết quả 664, 0.077 ms
greedy  : kết quả 664, 0.099 ms


## 12. Bộ test

### 12.1. Ví dụ đề và các trường hợp biên

In [18]:
tests = [
    ([1, 7, 4, 9, 2, 5], 6),
    ([1, 17, 5, 10, 13, 15, 10, 5, 16, 8], 7),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9], 2),
    ([5], 1),                           # một phần tử
    ([3, 3], 1),                        # hai phần tử bằng nhau
    ([3, 4], 2),                        # hai phần tử khác nhau
    ([0, 0, 0, 0, 0], 1),               # toàn bằng nhau
    ([9, 8, 7, 6, 5], 2),               # giảm dần
    ([1, 1, 7, 7, 4, 4, 9, 9], 4),      # các cặp bằng nhau liên tiếp
    ([0, 1000, 0, 1000, 0, 1000], 6),   # biên giá trị
    ([2, 1, 2, 1, 2, 1], 6),            # luân phiên sẵn
]
for nums, expected in tests:
    for cls in (SolutionN2, SolutionUpDown, SolutionGreedy, Solution):
        got = cls().wiggleMaxLength(nums)
        assert got == expected, (cls.__name__, nums, got, expected)
    assert wiggle_brute(nums) == expected
print("Tất cả test cơ bản đều đúng,", len(tests), "trường hợp")

Tất cả test cơ bản đều đúng, 11 trường hợp


### 12.2. Phản ví dụ cho các cách làm sai

1. **Đếm mọi cặp liền kề khác nhau** (quên điều kiện luân phiên): `[1,2,3,2]` phải ra `3` (bỏ `2`), nhưng cách sai cho `4`.
2. **Chỉ xét dãy con liên tiếp (subarray)**: `[1,5,2,2,6]`: dãy con `1,5,2,6` dài `4`; subarray lắc dài nhất chỉ `3`.

In [19]:
def dem_cap_khac_sai(nums):
    ans = 1
    for i in range(1, len(nums)):
        if nums[i] != nums[i - 1]:
            ans += 1
    return ans

def subarray_sai(nums):
    best = 1
    cur = 1
    for i in range(1, len(nums)):
        if i >= 2 and (nums[i] - nums[i - 1]) * (nums[i - 1] - nums[i - 2]) < 0:
            cur += 1
        elif nums[i] != nums[i - 1]:
            cur = 2
        else:
            cur = 1
        best = max(best, cur)
    return best

for nums in ([1, 1, 1], [1, 5, 2, 2, 6], [1, 2, 3, 2]):
    print(nums, "đúng:", Solution().wiggleMaxLength(nums), "| đếm cặp khác nhau:", dem_cap_khac_sai(nums), "| subarray:", subarray_sai(nums))
assert dem_cap_khac_sai([1, 2, 3, 2]) != Solution().wiggleMaxLength([1, 2, 3, 2])
assert subarray_sai([1, 5, 2, 2, 6]) != Solution().wiggleMaxLength([1, 5, 2, 2, 6])

[1, 1, 1] đúng: 1 | đếm cặp khác nhau: 1 | subarray: 1
[1, 5, 2, 2, 6] đúng: 4 | đếm cặp khác nhau: 4 | subarray: 3
[1, 2, 3, 2] đúng: 3 | đếm cặp khác nhau: 4 | subarray: 3


### 12.3. Đối chiếu ngẫu nhiên (có seed) — vét cạn và oracle độc lập

Ta dùng thêm một **oracle thứ hai** khác hẳn: đệ quy có nhớ `tiep(i, last, huong)` = số phần tử tối đa còn có thể nối thêm sau phần tử cuối có giá trị `last`, khi bước kế tiếp phải đi theo `huong`; thử mọi phần tử `j >= i` làm bước tiếp theo.

In [20]:
import functools

def oracle_memo(nums):
    n = len(nums)

    @functools.lru_cache(maxsize=None)
    def tiep(i, last, huong):         # huong: hướng bước kế tiếp mong muốn (+1/-1); -> số phần tử thêm tối đa
        best = 0
        for j in range(i, n):
            if (huong > 0 and nums[j] > last) or (huong < 0 and nums[j] < last):
                best = max(best, 1 + tiep(j + 1, nums[j], -huong))
        return best

    best = 1
    for s in range(n):
        best = max(best, 1 + tiep(s + 1, nums[s], 1), 1 + tiep(s + 1, nums[s], -1))
    return best

for nums, expected in tests:
    assert oracle_memo(nums) == expected, (nums, oracle_memo(nums), expected)
print("Oracle đệ quy khớp với các test cơ bản")

Oracle đệ quy khớp với các test cơ bản


In [21]:
random.seed(3760)
dem = 0
for _ in range(1500):
    n = random.randint(1, 12)
    nums = [random.randint(0, random.choice([1, 3, 6, 20])) for _ in range(n)]
    ket_qua = Solution().wiggleMaxLength(nums)
    assert ket_qua == wiggle_brute(nums), nums
    assert ket_qua == oracle_memo(nums), nums
    assert ket_qua == SolutionN2().wiggleMaxLength(nums) == SolutionGreedy().wiggleMaxLength(nums), nums
    dem += 1
print("Đối chiếu vét cạn + oracle đệ quy + 4 cách:", dem, "input ngẫu nhiên nhỏ - khớp")

Đối chiếu vét cạn + oracle đệ quy + 4 cách: 1500 input ngẫu nhiên nhỏ - khớp


In [22]:
random.seed(37600)
for _ in range(200):
    n = random.randint(1, 1000)
    nums = [random.randint(0, 1000) for _ in range(n)]
    a = SolutionN2().wiggleMaxLength(nums)
    b = SolutionUpDown().wiggleMaxLength(nums)
    c = SolutionGreedy().wiggleMaxLength(nums)
    assert a == b == c, (a, b, c)
    assert 1 <= a <= n
print("3 cách O(n^2)/O(n)/greedy khớp trên 200 input kích thước tới 1000")

3 cách O(n^2)/O(n)/greedy khớp trên 200 input kích thước tới 1000


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Coi hiệu `0` là một lần đổi chiều

Nếu `nums[i] == nums[i-1]` thì hiệu là `0`, **không** phải dương/âm. Đếm nó sẽ làm đáp án quá lớn.

In [23]:
def sai_1(nums):
    up = down = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            up = down + 1
        else:                                  # SAI: gộp cả trường hợp bằng vào "xuống"
            down = up + 1
    return max(up, down)

print("[5,5,5]: sai =", sai_1([5, 5, 5]), "| đúng =", Solution().wiggleMaxLength([5, 5, 5]))

[5,5,5]: sai = 2 | đúng = 1


Vì sao sai: `5 → 5` bị tính là bước xuống nên `down` tăng một cách vô lý. **Đúng:** dùng `elif nums[i] < nums[i-1]` và để trường hợp bằng nhau không làm gì.

### Lỗi 2: Nhầm với dãy con liên tiếp (subarray) hoặc chỉ so sánh `nums[i]` với `nums[i-1]` rồi **reset**

Đề cho phép **bỏ phần tử**, nên khi gặp đoạn tăng dài ta không "bắt đầu lại" mà **giữ nguyên** `up`.

In [24]:
def sai_2(nums):
    # Reset độ dài khi hai hiệu liên tiếp cùng dấu (coi như subarray)
    best = cur = 1
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        if i >= 2 and d * (nums[i - 1] - nums[i - 2]) < 0:
            cur += 1
        elif d != 0:
            cur = 2
        else:
            cur = 1
        best = max(best, cur)
    return best

nums = [1, 17, 5, 10, 13, 15, 10, 5, 16, 8]
print("sai =", sai_2(nums), "| đúng =", Solution().wiggleMaxLength(nums))

sai = 4 | đúng = 7


Đáp án đúng là `7` còn bản "subarray" chỉ ra `4`. **Đúng:** giữ `up`, `down` qua cả đoạn đơn điệu.

### Lỗi 3 (tham lam): cập nhật `prev` cả khi hiệu là `0`

Với `[1, 2, 2, 3]`: các hiệu `1, 0, 1`. Nếu đặt `prev = diff` ở mọi bước, sau hiệu `0` ta có `prev = 0`, rồi hiệu `1` lại bị tính như "hiệu đầu tiên" → đếm thừa.

In [25]:
def sai_3(nums):
    if len(nums) < 2:
        return len(nums)
    ans, prev = 1, 0
    for i in range(1, len(nums)):
        diff = nums[i] - nums[i - 1]
        if (diff > 0 and prev <= 0) or (diff < 0 and prev >= 0):
            ans += 1
        prev = diff                         # SAI: luôn cập nhật, kể cả khi diff == 0
    return ans

print("[1,2,2,3]: sai =", sai_3([1, 2, 2, 3]), "| đúng =", Solution().wiggleMaxLength([1, 2, 2, 3]))

[1,2,2,3]: sai = 3 | đúng = 2


**Đúng:** chỉ `prev = diff` khi đã đếm một lần đổi chiều (như Cách 4 ở mục 7).

### Lỗi 4: Quên trường hợp `n = 1` / bắt đầu `ans = 2`

Nếu khởi tạo `ans = 2` (nghĩ "ít nhất hai phần tử") thì `[7]` và `[3,3]` ra sai.

In [26]:
def sai_4(nums):
    ans, prev = 2, 0                        # SAI: giả sử luôn >= 2
    for i in range(1, len(nums)):
        diff = nums[i] - nums[i - 1]
        if (diff > 0 and prev < 0) or (diff < 0 and prev > 0):
            ans += 1
        if diff != 0:
            prev = diff
    return ans

for nums in ([7], [3, 3], [1, 2, 3]):
    print(nums, "sai =", sai_4(nums), "| đúng =", Solution().wiggleMaxLength(nums))

[7] sai = 2 | đúng = 1
[3, 3] sai = 2 | đúng = 1
[1, 2, 3] sai = 2 | đúng = 2


**Đúng:** khởi tạo `up = down = 1` (hoặc `ans = 1` và xử lý hiệu khác 0 đầu tiên).

### Lỗi 5: Quên trả `max(up, down)`

Chỉ trả `up` hoặc `down` thì sai khi dãy tối ưu kết thúc bằng bước ngược lại.

In [27]:
def sai_5(nums):
    up = down = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            up = down + 1
        elif nums[i] < nums[i - 1]:
            down = up + 1
    return up                                # SAI: thiếu max(up, down)

nums = [1, 17, 5, 10, 13, 15, 10, 5, 16, 8]
print("sai =", sai_5(nums), "| đúng =", Solution().wiggleMaxLength(nums))

sai = 6 | đúng = 7


## 14. Mẹo & nhận dạng pattern

- **Từ khóa:** "luân phiên", "dãy con dài nhất", "hiệu đổi dấu" → thử **DP hai trạng thái (lên/xuống)**; nếu đề không cần dựng lại dãy thì thử rút gọn thành hai biến hoặc **tham lam đếm đỉnh/đáy**.
- **Họ bài:** *Longest Increasing Subsequence* (300; DP `O(n²)`), *Longest Turbulent Subarray* (978; subarray), *Wiggle Sort* (280, 324), *Best Time to Buy and Sell Stock II* (122; cộng các đoạn tăng — cũng là "đếm đỉnh/đáy").
- **Khi hiệu bằng `0`:** coi như phần tử đó **không tồn tại** (có thể bỏ khỏi dãy con mà không mất gì).
- **Mẹo kiểm tra:** dãy tăng chặt/giảm chặt dài bao nhiêu cũng chỉ cho đáp án `2`; mảng toàn bằng nhau cho `1`.
- **Hai biến `up`/`down` = máy trạng thái hai ô.** Nhận ra DP `dp[i]` chỉ phụ thuộc `dp[i-1]` là chiêu rút gọn bộ nhớ chung.

## 15. Biến thể

### 15.1. In ra một dãy lắc dài nhất (không chỉ độ dài)

Từ mục 4: giữ các **cực trị** (hai đầu mỗi đoạn đơn điệu). Cài đặt: duyệt, nếu hướng đổi (hoặc là hiệu khác 0 đầu tiên) thì thêm phần tử; nếu hướng giữ nguyên thì **thay phần tử cuối** bằng phần tử mới (kéo dài đoạn).

In [28]:
def wiggle_sequence(nums):
    if not nums:
        return []
    res = [nums[0]]
    prev = 0
    for i in range(1, len(nums)):
        diff = nums[i] - nums[i - 1]
        if diff == 0:
            continue
        if (diff > 0 and prev <= 0) or (diff < 0 and prev >= 0):
            res.append(nums[i])
            prev = diff
        else:
            res[-1] = nums[i]               # cùng chiều: kéo dài đoạn, thay phần tử cuối
    return res

for nums, expected in examples:
    seq = wiggle_sequence(nums)
    print(nums, "->", seq)
    assert is_wiggle(seq) and len(seq) == expected
    # seq phải là dãy con của nums
    it = iter(nums)
    assert all(x in it for x in seq)

[1, 7, 4, 9, 2, 5] -> [1, 7, 4, 9, 2, 5]
[1, 17, 5, 10, 13, 15, 10, 5, 16, 8] -> [1, 17, 5, 15, 5, 16, 8]
[1, 2, 3, 4, 5, 6, 7, 8, 9] -> [1, 9]


### 15.2. Dãy con lắc dài nhất theo kiểu "không chặt" (cho phép hiệu `0`)

Nếu cho phép các hiệu `0` **ở bất kỳ vị trí nào** thì bài toán gần như tầm thường (có thể lấy cả dãy). Vì vậy đề gốc cố ý yêu cầu **chặt**.

### 15.3. Subarray lắc dài nhất (LeetCode 978 — Longest Turbulent Subarray)

Bắt buộc **liên tiếp**: khi hai hiệu liền cùng dấu hoặc có `0` thì phải "reset". Ta sẽ làm ở bài tập 5.

### 15.4. Các biến thể khác

- Đếm **số** dãy con lắc dài nhất (DP đếm kèm `cnt`).
- Dãy con lắc dài nhất **bắt đầu bằng bước lên** (chỉ lấy `up` ở một nhánh).
- Có **trọng số**: tổng lớn nhất của dãy lắc → DP `O(n²)` hoặc cấu trúc dữ liệu (cây phân đoạn) — Cách 2 mở rộng được còn tham lam thì không.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi dãy con | `O(2^n · n)` | `O(n)` | Oracle kiểm chứng `n ≤ 15` |
| 2. DP `up[]`/`down[]` | Dài nhất kết thúc tại `i` với bước cuối lên/xuống | `O(n²)` | `O(n)` | Khi có biến thể có trọng số / cần dựng lại |
| 3. DP hai biến | Rút gọn Cách 2 | `O(n)` | `O(1)` | Lời giải nộp, trả lời follow up |
| 4. Tham lam | Đếm đỉnh/đáy | `O(n)` | `O(1)` | Khi muốn trực giác hình học, in dãy |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `wiggleMaxLength([4])`; (b) `wiggleMaxLength([4, 4, 4])`; (c) `wiggleMaxLength([1, 3, 2])`; (d) `sign(-7)`; (e) `is_wiggle([2, 5, 1, 1])`.

**Bài 2 (tính tay).** Với `nums = [3, 3, 3, 2, 5, 5, 1]`, viết dãy hiệu, bảng `up`/`down` theo từng `i` (như mục 8) và độ dài tối ưu.

**Bài 3 (điền TODO).** Hoàn thành `wiggle_todo(nums)` theo Cách 3.

In [29]:
def wiggle_todo(nums):
    # TODO 1: khởi tạo up = down = 1
    # TODO 2: for i in range(1, len(nums)):
    #             nếu nums[i] > nums[i-1]: up = down + 1
    #             nếu nums[i] < nums[i-1]: down = up + 1   (bằng nhau: bỏ qua)
    # TODO 3: return max(up, down)
    return None

# KHÔNG gọi wiggle_todo ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `wiggle_greedy2(nums)` theo Cách 4 (hiệu khác 0 gần nhất `prev`, đếm đổi chiều) **mà không nhìn lại** mục 7. Kiểm tra với `[1,7,4,9,2,5] → 6` và `[1,2,2,3] → 2`.

**Bài 5 (biến thể / mini challenge).** Viết `turbulent_subarray(nums)`: độ dài **subarray** (liên tiếp) lắc dài nhất (LeetCode 978). Kiểm tra: `[9,4,2,10,7,8,8,1,9] → 5`, `[4,8,12,16] → 2`, `[100] → 1`. Gợi ý: so với Cách 3, khi hai hiệu cùng dấu hoặc hiệu `0` thì **đặt lại** độ dài về `2` hoặc `1` thay vì giữ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `1`, (b) `1`, (c) `3`, (d) `-1`, (e) `False` (hiệu cuối là `0`). Bài 3: nhớ `elif`, không phải `else`.
- **Mức 2 (vừa):** Bài 2: các hiệu `0, 0, -1, 3, 0, -4`; bỏ hiệu `0` ta còn `-1, 3, -4` luân phiên → `1 + 3 = 4`. Bài 4: `ans = 1`, `prev = 0`; chỉ cập nhật `prev` trong nhánh `if` đổi chiều.
- **Mức 3 (sâu):** Bài 5: duy trì `cur` = độ dài subarray lắc kết thúc tại `i`: nếu `nums[i] == nums[i-1]` thì `cur = 1`; nếu `i >= 2` và hiệu đổi dấu so với hiệu trước thì `cur += 1`; còn lại `cur = 2`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
assert Solution().wiggleMaxLength([4]) == 1
assert Solution().wiggleMaxLength([4, 4, 4]) == 1
assert Solution().wiggleMaxLength([1, 3, 2]) == 3
assert sign(-7) == -1
assert is_wiggle([2, 5, 1, 1]) is False
print("Bài 1 OK")

# Bài 2
nums = [3, 3, 3, 2, 5, 5, 1]
assert [b - a for a, b in zip(nums, nums[1:])] == [0, 0, -1, 3, 0, -4]
assert Solution().wiggleMaxLength(nums) == 4 == wiggle_brute(nums)
print("Bài 2 OK")

# Bài 3
def wiggle_todo_ans(nums):
    up = down = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            up = down + 1
        elif nums[i] < nums[i - 1]:
            down = up + 1
    return max(up, down)

random.seed(1)
for _ in range(300):
    arr = [random.randint(0, 9) for _ in range(random.randint(1, 12))]
    assert wiggle_todo_ans(arr) == wiggle_brute(arr)
print("Bài 3 OK")

# Bài 4
def wiggle_greedy2(nums):
    if len(nums) < 2:
        return len(nums)
    ans, prev = 1, 0
    for i in range(1, len(nums)):
        diff = nums[i] - nums[i - 1]
        if (diff > 0 and prev <= 0) or (diff < 0 and prev >= 0):
            ans += 1
            prev = diff
    return ans

assert wiggle_greedy2([1, 7, 4, 9, 2, 5]) == 6 and wiggle_greedy2([1, 2, 2, 3]) == 2
for _ in range(300):
    arr = [random.randint(0, 9) for _ in range(random.randint(1, 12))]
    assert wiggle_greedy2(arr) == wiggle_brute(arr)
print("Bài 4 OK")

# Bài 5
def turbulent_subarray(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        d = nums[i] - nums[i - 1]
        if d == 0:
            cur = 1
        elif i >= 2 and d * (nums[i - 1] - nums[i - 2]) < 0:
            cur += 1
        else:
            cur = 2
        best = max(best, cur)
    return best

def turbulent_brute(nums):
    n, best = len(nums), 1
    for i in range(n):
        for j in range(i, n):
            if is_wiggle(nums[i:j + 1]):
                best = max(best, j - i + 1)
    return best

assert turbulent_subarray([9, 4, 2, 10, 7, 8, 8, 1, 9]) == 5
assert turbulent_subarray([4, 8, 12, 16]) == 2 and turbulent_subarray([100]) == 1
for _ in range(300):
    arr = [random.randint(0, 5) for _ in range(random.randint(1, 10))]
    assert turbulent_subarray(arr) == turbulent_brute(arr), arr
print("Bài 5 OK")

Bài 1 OK
Bài 2 OK
Bài 3 OK


Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (DP hai biến):**

```python
up = down = 1
for i in range(1, len(nums)):
    if nums[i] > nums[i - 1]:
        up = down + 1
    elif nums[i] < nums[i - 1]:
        down = up + 1
return max(up, down)
```

**Checklist:**

- [ ] Khởi tạo `up = down = 1` (chạy đúng với `n = 1`).
- [ ] Dùng `elif`, không dùng `else`: hiệu `0` **không** thay đổi trạng thái.
- [ ] `up = down + 1` (lấy từ trạng thái **đối diện**), không phải `up + 1`.
- [ ] Cuối cùng trả `max(up, down)`.
- [ ] Nếu viết bản tham lam: chỉ cập nhật `prev` khi có đổi chiều thật sự.
- [ ] Thử `[5]`, `[3,3]`, `[1,2,3,4,5]`, `[1,7,4,9,2,5]` trước khi nộp.
- [ ] Đúng signature `class Solution` / `wiggleMaxLength(self, nums: List[int]) -> int`.